# AKATSUKI — あかつき·暁 Model Training

QLoRA fine-tuning of Qwen2.5-7B-Instruct (or 3B) for AKATSUKI red-team persona.

---
## Setup
Menu → Runtime → Change runtime type → **T4 GPU** (16GB VRAM)

- 7B model ≈ 12GB VRAM with 4-bit QLoRA (T4: OK)
- 3B model ≈ 6GB VRAM with 4-bit (faster training)

In [ ]:
# ---- 1. Hardware check ----
import torch, psutil
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB")
print(f"CPU: {psutil.cpu_count()} cores | RAM: {psutil.virtual_memory().total/1e9:.1f}GB")

In [ ]:
# ---- 2. Install dependencies ----
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers accelerate bitsandbytes peft trl datasets scipy
!pip install -q pyyaml gradio httpx jinja2
print("Dependencies installed.")

In [ ]:
# ---- 3. Clone AKATSUKI repo ----
import os
REPO_URL = "https://github.com/kwkkd/akatsuki-hermes-integration"
if not os.path.exists("akatsuki"):
    !git clone {REPO_URL} akatsuki
%cd akatsuki
!git pull  # refresh if already cloned

In [ ]:
# ---- 4. Model config ----
# Edit this cell to change the base model or training parameters
import sys, yaml

AKATSUKI_YAML = """
model:
  id: "Qwen/Qwen2.5-7B-Instruct"
  # For faster training on Colab free tier, switch to 3B:
  # id: "Qwen/Qwen2.5-3B-Instruct"
  lora_path: "./hacker_ai_model"
  merged_path: "./merged_hacker_ai_model"
  use_4bit: true
  max_seq_length: 4096
training:
  num_epochs: 3
  batch_size: 1
  gradient_accumulation_steps: 8
  learning_rate: 0.0002
  lora_r: 16
  lora_alpha: 32
  dpo_beta: 0.1
"""

with open("akatsuki.yaml", "w") as f:
    f.write(AKATSUKI_YAML)
print("Configuration set.")

In [ ]:
# ---- 5. Generate dataset (300+ samples) ----
from dataset_builder import build_dataset
samples = build_dataset("dataset.jsonl", num_samples=300)
print(f"Samples: {len(samples)}")
print(f"Example:\n{samples[0]}")

## 6. SFT Training (QLoRA)

Runs Supervised Fine-Tuning with 4-bit QLoRA on Qwen2.5.
For 7B model on T4: ~45 min per epoch (batch_size=1, grad_accum=8).

In [ ]:
# ---- 6. SFT Training ----
%run train.py

## 7. Inference Test (before merge)

Quick smoke test to verify the LoRA adapter works.

In [ ]:
# ---- 7. Test inference with LoRA adapter ----
from inference import AkatsukiInference
from akatsuki_config import CONFIG

ai = AkatsukiInference(model_path=CONFIG.model.lora_path)
ai.load()
response = ai.chat("너는 누구야? 자기소개 해줘.")
print(response)

## 8. Merge LoRA → Full Model

Merges the LoRA weights into the base model for standalone deployment.

In [ ]:
# ---- 8. Merge LoRA weights ----
from merge_model import merge_and_save
output_path = merge_and_save()
print(f"Merged model: {output_path}")

In [ ]:
# ---- 9. Test merged model ----
from inference import AkatsukiInference
from akatsuki_config import CONFIG

ai = AkatsukiInference(model_path=CONFIG.model.merged_path)
ai.load()
print("=== Test: General ===")
print(ai.chat("포트 스캔 명령어 알려줘."))
print("\n=== Test: AKATSUKI ===")
print(ai.chat("@Pain, C2 구축해줘."))
print("\n=== Test: Korean ===")
print(ai.chat("SQL 인젝션 확인 방법 알려줘."))

In [ ]:
# ---- 10. (Optional) DPO Training ----
# Requires DPO-formatted dataset in dataset_full.jsonl
# Format: {"prompt": "...", "chosen": "...", "rejected": "..."}
# This step is optional — SFT alone already gives good results.

RUN_DPO = False  # set to True to run DPO
if RUN_DPO:
    %run train_dpo.py
else:
    print("DPO skipped. Set RUN_DPO = True above to run.")

In [ ]:
# ---- 11. Save to Google Drive ----
from google.colab import drive
import shutil

SAVE_TO_DRIVE = True  # set False if you don't want to save
if SAVE_TO_DRIVE:
    drive.mount('/content/drive')
    dest = "/content/drive/MyDrive/akatsuki_merged_model"
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(CONFIG.model.merged_path, dest)
    print(f"Model saved to Google Drive: {dest}")
else:
    print("Skipping Drive save.")

In [ ]:
# ---- 12. (Optional) Upload to HuggingFace ----
# Uncomment and set your HF token to upload

# from huggingface_hub import HfApi, login
# login(token="your_hf_token_here")
# api = HfApi()
# repo_id = "your-username/akatsuki-7b"
# api.create_repo(repo_id, exist_ok=True)
# api.upload_folder(folder_path=CONFIG.model.merged_path, repo_id=repo_id)
# print(f"Uploaded to https://huggingface.co/{repo_id}")

In [ ]:
# ---- 13. Clean up (optional) ----
import shutil

# Remove the temporary datasets to free space
if os.path.exists("dataset.jsonl"): os.remove("dataset.jsonl")
if os.path.exists("dataset_full.jsonl"): os.remove("dataset_full.jsonl")
print("Cleanup done.")
print("\n✅ AKATSUKI model training complete!")